# Data Fusion — Evaluation Set Creation

### Workflow

1. **Part 1** — build airport groups ("clusters") from the identity-resolution correspondences.
2. **Part 2** — per cluster/attribute, flag which attributes are *comparable* (≥2 sources have a
   value) and which of those actually *disagree* after normalization; summarize per attribute.
3. **Part 3** — pick candidate clusters to manually verify, prioritizing clusters that cover
   several disagreeing attributes at once, then back-fill any attribute that's still
   under-represented.
4. **Part 3b** — manual check (outside this notebook's code): confirm the picked clusters are
   really the same airport and that their ICAO is correct.
5. **Part 4** — *(not yet written)* look up external ground-truth values (Overpass, Nominatim,
   timezonefinder) for the manually-confirmed `(id, icao)` pairs and write `gold.xml`.

## Part 1 — Build Clusters from Correspondences

**Goal**: group matched records across the three sources into one row per real-world airport,
so Part 2 can compare each attribute's values across sources within a group.

**Method**: there are only two correspondence files, both one-to-one and both anchored on
OurAirports (`wd_oa_correspondences.csv` is wd↔oa, `correspondences_oa_kg.csv` is oa↔kg; there is
no direct wd↔kg matcher). So every cluster has exactly one `oa_id`, and 0 or 1 `wd_id` / `kg_id` —
a plain two-way left join on `oa_id` is enough, no need for a general union-find.

In [1]:
from pathlib import Path

import pandas as pd

CORR_DIR = Path("../../data/identity_resolution/outputs")
OUTPUT_DIR = Path("../../data/data_fusion")
GOLD_DIR = OUTPUT_DIR / "golden_standard"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

wd_oa = pd.read_csv(CORR_DIR / "wd_oa_correspondences.csv")[["id1", "id2"]].rename(
    columns={"id1": "wd_id", "id2": "oa_id"}
)
oa_kg = pd.read_csv(CORR_DIR / "correspondences_oa_kg.csv")[["id1", "id2"]].rename(
    columns={"id1": "oa_id", "id2": "kg_id"}
)

print(f"wd-oa correspondences: {len(wd_oa):,}")
print(f"oa-kg correspondences: {len(oa_kg):,}")

wd-oa correspondences: 22,982
oa-kg correspondences: 5,818


In [2]:
# Sanity check: both correspondence files must be one-to-one (no oa_id matched more than
# once within either file), otherwise the join below would silently fan out rows.
assert not wd_oa["wd_id"].duplicated().any(), "a wd_id matched to more than one oa_id"
assert not wd_oa["oa_id"].duplicated().any(), "an oa_id matched to more than one wd_id"
assert not oa_kg["oa_id"].duplicated().any(), "an oa_id matched to more than one kg_id"
assert not oa_kg["kg_id"].duplicated().any(), "a kg_id matched to more than one oa_id"
print("one-to-one check passed for both correspondence files")

one-to-one check passed for both correspondence files


In [3]:
# Every cluster has an oa_id (oa is the hub of both matchers): take the union of oa_ids that
# appear in either correspondence file, then attach the matched wd_id / kg_id (NaN if absent).
all_oa_ids = pd.Index(sorted(set(wd_oa["oa_id"]) | set(oa_kg["oa_id"])), name="oa_id")

clusters = (
    pd.DataFrame(index=all_oa_ids)
    .reset_index()
    .merge(wd_oa, on="oa_id", how="left")
    .merge(oa_kg, on="oa_id", how="left")
)

assert len(clusters) == len(all_oa_ids), "join produced extra rows: oa_id was not unique somewhere"

clusters.insert(0, "cluster_id", range(len(clusters)))
clusters["source_combination"] = (
    clusters["wd_id"].notna().map({True: "wd", False: ""})
    + clusters["oa_id"].notna().map({True: "+oa", False: ""})
    + clusters["kg_id"].notna().map({True: "+kg", False: ""})
).str.lstrip("+")

print(f"Total clusters: {len(clusters):,}")
print(clusters["source_combination"].value_counts())
clusters.head()

Total clusters: 23,629
source_combination
wd+oa       17811
wd+oa+kg     5171
oa+kg         647
Name: count, dtype: int64


,cluster_id,oa_id,wd_id,kg_id,source_combination
0,0,oa_10000,wd_17606,NaN,wd+oa
1,1,oa_10001,NaN,kg_5650,oa+kg
2,2,oa_10005,wd_17745,NaN,wd+oa
3,3,oa_10006,wd_18873,NaN,wd+oa
4,4,oa_10008,wd_18234,NaN,wd+oa


## Part 2 — Comparable vs. Disagreeing Attributes

**Goal**: for every cluster, figure out — per attribute — whether fusion actually had to resolve
anything, and if so, whether the resolution is trivial (sources happen to agree) or meaningful
(sources disagree, so the conflict-resolution rule matters).

**Method**: two layers of flags per cluster × attribute:
- **comparable**: ≥2 sources have a non-missing value for that attribute (same test as before;
  blank/whitespace-only strings count as missing).
- **disagree**: only defined where comparable is True — the non-missing values, after
  normalization, are not all the same.
  - `name` / `city` / `countryName` (free text): lowercase, strip, collapse whitespace, strip
    punctuation.
  - `icao` / `iata` / `localCode` / `countryCode` / `timezone` (codes/identifiers): lowercase,
    strip, collapse whitespace only — punctuation can be meaningful here (e.g. `America/New_York`).
  - `lat` / `lon` (numeric): disagree if the values span more than 0.0001° (~11 m at the equator;
    a per-coordinate degree difference, not true geodesic distance — an acceptable simplification
    since none of these airports are near the poles).
  - `elevation` (numeric): disagree if the values span more than 5 m.

Each attribute is only pulled from the sources whose normalized CSV actually has that column
(e.g. Kaggle has no `elevation`/`localCode` column at all), so the denominator for each attribute
reflects its real source coverage.

In [4]:
import re

from PyDI.io import load_csv

NORM_DIR = Path("../../data/normalized_outputs")

df_wd = load_csv(NORM_DIR / "wikidataNormalized.csv", name="wikidata")
df_oa = load_csv(NORM_DIR / "ourairportsNormalized.csv", name="ourairports")
df_kg = load_csv(NORM_DIR / "kaggleNormalized.csv", name="kaggle")

SOURCE_DFS = {"wd": df_wd, "oa": df_oa, "kg": df_kg}

EVAL_ATTRS = [
    "icao", "iata", "name", "lat", "lon", "city",
    "elevation", "timezone", "localCode", "countryCode", "countryName",
]

# Free-text attributes get punctuation stripped before comparison; codes/identifiers don't.
FREE_TEXT_ATTRS = {"name", "city", "countryName"}
NUMERIC_TOLERANCE = {"lat": 0.0001, "lon": 0.0001, "elevation": 5.0}

ATTR_SOURCES = {
    attr: [src for src, df in SOURCE_DFS.items() if attr in df.columns]
    for attr in EVAL_ATTRS
}
for attr, sources in ATTR_SOURCES.items():
    print(f"{attr:<12} {sources}")

icao         ['wd', 'oa', 'kg']
iata         ['wd', 'oa', 'kg']
name         ['wd', 'oa', 'kg']
lat          ['wd', 'oa', 'kg']
lon          ['wd', 'oa', 'kg']
city         ['wd', 'oa', 'kg']
elevation    ['wd', 'oa']
timezone     ['wd', 'kg']
localCode    ['wd', 'oa']
countryCode  ['wd', 'oa', 'kg']
countryName  ['wd', 'oa', 'kg']


In [5]:
# Pull each attribute's raw value from every source that has it, one column per (attr, source).
values = clusters[["cluster_id", "wd_id", "oa_id", "kg_id", "source_combination"]].copy()

for attr, sources in ATTR_SOURCES.items():
    for src in sources:
        lookup = SOURCE_DFS[src].set_index("id")[attr]
        values[f"{attr}_{src}"] = values[f"{src}_id"].map(lookup)

values.head()

,cluster_id,wd_id,oa_id,kg_id,source_combination,icao_wd,icao_oa,icao_kg,iata_wd,iata_oa,...,timezone_wd,timezone_kg,localCode_wd,localCode_oa,countryCode_wd,countryCode_oa,countryCode_kg,countryName_wd,countryName_oa,countryName_kg
0,0,wd_17606,oa_10000,NaN,wd+oa,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,80A,80A,US,US,NaN,united states,united states,NaN
1,1,NaN,oa_10001,kg_5650,oa+kg,NaN,NaN,NaN,NaN,AOS,...,NaN,America/Anchorage,NaN,AK81,NaN,US,US,NaN,united states,united states of america
2,2,wd_17745,oa_10005,NaN,wd+oa,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,AK85,AK85,US,US,NaN,united states,united states,NaN
3,3,wd_18873,oa_10006,NaN,wd+oa,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,AK86,AK86,US,US,NaN,united states,united states,NaN
4,4,wd_18234,oa_10008,NaN,wd+oa,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,AK88,AK88,US,US,NaN,united states,united states,NaN


In [6]:
def is_missing(v) -> bool:
    if pd.isna(v):
        return True
    if isinstance(v, str) and v.strip() == "":
        return True
    return False


def normalize_string(v, strip_punctuation: bool) -> str:
    s = str(v).strip().lower()
    s = re.sub(r"\s+", " ", s)
    if strip_punctuation:
        s = re.sub(r"[^\w\s]", "", s)
    return s


def attribute_flags(row, attr, sources):
    raw = [row[f"{attr}_{src}"] for src in sources]
    present = [v for v in raw if not is_missing(v)]

    comparable = len(present) >= 2
    if not comparable:
        return comparable, pd.NA

    if attr in NUMERIC_TOLERANCE:
        nums = [float(v) for v in present]
        disagree = (max(nums) - min(nums)) > NUMERIC_TOLERANCE[attr]
    else:
        normed = {normalize_string(v, attr in FREE_TEXT_ATTRS) for v in present}
        disagree = len(normed) > 1

    return comparable, disagree


for attr, sources in ATTR_SOURCES.items():
    flags = values.apply(lambda row: attribute_flags(row, attr, sources), axis=1)
    values[f"{attr}_comparable"] = flags.map(lambda t: t[0])
    values[f"{attr}_disagree"] = flags.map(lambda t: t[1])


### Table 1 — comparable vs. disagree counts per attribute

In [7]:
summary = pd.DataFrame({
    "attribute": EVAL_ATTRS,
    "sources": [", ".join(ATTR_SOURCES[a]) for a in EVAL_ATTRS],
    "comparable": [values[f"{a}_comparable"].sum() for a in EVAL_ATTRS],
    "disagree": [(values[f"{a}_disagree"] == True).sum() for a in EVAL_ATTRS],  # noqa: E712 (NA-safe)
})
summary["disagree_rate"] = (summary["disagree"] / summary["comparable"]).round(3)
summary = summary.sort_values("disagree", ascending=False).reset_index(drop=True)

LOW_SAMPLE_THRESHOLD = 10
summary["flag"] = summary["disagree"].apply(
    lambda n: "insufficient disagree samples" if n < LOW_SAMPLE_THRESHOLD else ""
)

summary

,attribute,sources,comparable,disagree,disagree_rate,flag
0,city,"wd, oa, kg",22954,20807,0.906,
1,lon,"wd, oa, kg",23629,17512,0.741,
2,lat,"wd, oa, kg",23629,15811,0.669,
3,name,"wd, oa, kg",23628,8678,0.367,
4,countryName,"wd, oa, kg",23624,3434,0.145,
5,elevation,"wd, oa",18107,1669,0.092,
6,timezone,"wd, kg",2422,1425,0.588,
7,icao,"wd, oa, kg",8202,243,0.030,
8,countryCode,"wd, oa, kg",23582,228,0.010,
9,iata,"wd, oa, kg",7928,62,0.008,


### Table 2 — disagree counts per attribute × source combination

In [8]:
rows = []
for attr in EVAL_ATTRS:
    disagreeing = values[values[f"{attr}_disagree"] == True]  # noqa: E712
    counts = disagreeing["source_combination"].value_counts()
    rows.append({
        "attribute": attr,
        **{combo: int(counts.get(combo, 0)) for combo in ["wd+oa", "oa+kg", "wd+oa+kg"]},
    })

by_combination = pd.DataFrame(rows).set_index("attribute")
by_combination["total"] = by_combination.sum(axis=1)
by_combination.sort_values("total", ascending=False)

,wd+oa,oa+kg,wd+oa+kg,total
attribute,,,,
city,16077,361,4369,20807
lon,13045,416,4051,17512
lat,11669,377,3765,15811
name,4194,474,4010,8678
countryName,1083,230,2121,3434
elevation,1178,0,491,1669
timezone,0,0,1425,1425
icao,51,20,172,243
countryCode,54,13,161,228


**Findings**:
- `lat`/`lon` disagree on 67–74% of comparable clusters even at an 11 m tolerance — consistent
  with the identity-resolution gold standard's own finding that these sources' coordinates are
  often several km apart for the *same* airport (noisy GPS entry, not a different airport). So for
  `lat`/`lon`, "disagree" is the normal case, not the exception.
- `city` disagrees on 90.6% of comparable clusters — much higher than expected. Spot-checking
  confirms the cause documented in the project README: Wikidata's `city` (from `located_in`) is
  often a state/region/country rather than an actual city (e.g. `city_wd="alaska"` while
  `city_oa="allakaket"`), and Kaggle's `city` sometimes appends the state (`"groton connecticut"`
  vs OurAirports' `"groton"`). This isn't 3-way noise so much as Wikidata's attribute being the
  wrong granularity.
- `timezone` disagrees on 58.8% of its (much smaller, wd+kg only) comparable set. Wikidata's raw
  values are a messy mix of IANA names, UTC offsets, and vague labels (`"UTC−07:00"`,
  `"Mountain Time Zone"`, `"America/Los Angeles"` with a space instead of `_`), while Kaggle is
  consistently proper IANA names (`"America/Denver"`). So this is a real representational
  inconsistency in Wikidata, not a formatting artifact our normalizer should paper over.
- `icao`, `iata`, `countryCode`, `localCode` disagree on <3% of comparable clusters — as expected
  for near-authoritative identifier/code fields.
- `localCode` has only 40 disagreeing clusters total (and 0 in oa+kg, since Kaggle doesn't carry
  it at all) — flagged by the `LOW_SAMPLE_THRESHOLD` check as having a thin sample; may need to
  take all 40 rather than a subset in Part 3.


## Part 3 — Sample Candidate Clusters

**Goal**: pick a fixed total of **125** clusters to manually verify (Part 3b), covering every
attribute's disagreeing cases well enough to evaluate fusion on it, without wasting the budget on
redundant single-purpose picks.

**Sample Method**: scarcest-pool-first greedy, with a floor + global top-up:
1. Rank attributes by disagree-pool size, ascending (`localCode` first, `city` last) — process the
   scarcest pools first, since they have the fewest options.
2. For each attribute, give it a floor of **`MIN_PER_ATTR = 8`** clusters: count how many are
   already selected (free, via overlap with a previous attribute's picks) and top up the rest from
   the attribute's own disagree pool, preferring clusters that also disagree on the *most other*
   attributes (`n_disagree_attrs`), and trying to cover each available `source_combination` at
   least once before filling the rest by that same ranking.
3. 11 attributes × 8 = 88 ≤ 125, so there's slack left (37 clusters) after every attribute has its
   floor. Spend it with a global top-up: keep adding the highest-`n_disagree_attrs` cluster not
   yet selected until the total reaches 125. This naturally favors the large, overlapping pools
   (`lat`/`lon`/`city`/`name`), since that's where multi-attribute clusters are most common.
4. Report, per attribute, how many of the final 125 actually disagree on it — the floor guarantees
   ≥8, but overlap usually pushes several attributes well above that for free.

In [9]:
import numpy as np

SEED_PART3 = 1234
rng = np.random.default_rng(SEED_PART3)
values["tiebreak"] = rng.random(len(values))

disagree_cols = [f"{attr}_disagree" for attr in EVAL_ATTRS]
values["n_disagree_attrs"] = sum(
    (values[col] == True).fillna(False).astype(int) for col in disagree_cols  # noqa: E712
)

MIN_PER_ATTR = 8 
TARGET_TOTAL = 125 

# scarcest disagree-pool first
attr_order = summary.sort_values("disagree")["attribute"].tolist()
print("Attribute processing order (scarcest pool first):")
print(attr_order)

Attribute processing order (scarcest pool first):
['localCode', 'iata', 'countryCode', 'icao', 'timezone', 'elevation', 'countryName', 'name', 'lat', 'lon', 'city']


In [10]:
def rank(df):
    """Highest n_disagree_attrs first; fixed-seed random tiebreak for determinism."""
    return df.sort_values(["n_disagree_attrs", "tiebreak"], ascending=[False, True])


def pick_for_attribute(need, pool):
    """Pick `need` rows from `pool`, trying one per source_combination first, then by rank."""
    picks = []
    for combo in pool["source_combination"].unique():
        if need <= 0:
            break
        combo_pool = rank(pool[pool["source_combination"] == combo])
        if combo_pool.empty:
            continue
        picks.append(combo_pool.iloc[0])
        pool = pool[pool["cluster_id"] != combo_pool.iloc[0]["cluster_id"]]
        need -= 1
    if need > 0:
        picks.extend(row for _, row in rank(pool).head(need).iterrows())
    return picks


selected: set[int] = set()

for attr in attr_order:
    candidates = values[values[f"{attr}_disagree"] == True]  # noqa: E712
    already_free = candidates["cluster_id"].isin(selected).sum()
    need = MIN_PER_ATTR - already_free
    if need <= 0:
        continue
    pool = candidates[~candidates["cluster_id"].isin(selected)]
    for row in pick_for_attribute(need, pool):
        selected.add(row["cluster_id"])

print(f"After per-attribute floor (min {MIN_PER_ATTR} each): {len(selected)} clusters selected")

After per-attribute floor (min 8 each): 23 clusters selected


In [11]:
if len(selected) < TARGET_TOTAL:
    remaining_pool = values[
        (values["n_disagree_attrs"] > 0) & (~values["cluster_id"].isin(selected))
    ]
    top_up = rank(remaining_pool).head(TARGET_TOTAL - len(selected))
    selected.update(top_up["cluster_id"].tolist())

print(f"After global top-up toward {TARGET_TOTAL}: {len(selected)} clusters selected")

After global top-up toward 125: 125 clusters selected


In [12]:
candidates_final = (
    values[values["cluster_id"].isin(selected)]
    .sort_values("cluster_id")
    .reset_index(drop=True)
)

### Coverage check

In [13]:
coverage = pd.DataFrame({
    "attribute": EVAL_ATTRS,
    "hit_count": [int((candidates_final[f"{a}_disagree"] == True).sum()) for a in EVAL_ATTRS],  # noqa: E712
})
coverage["meets_min"] = coverage["hit_count"] >= MIN_PER_ATTR
coverage = coverage.sort_values("hit_count").reset_index(drop=True)
coverage

,attribute,hit_count,meets_min
0,localCode,8,True
1,iata,18,True
2,icao,26,True
3,elevation,54,True
4,countryCode,58,True
5,timezone,89,True
6,countryName,101,True
7,name,120,True
8,lon,123,True
9,lat,124,True


In [14]:
candidates_final["source_combination"].value_counts()

source_combination
wd+oa+kg    103
wd+oa        18
oa+kg         4
Name: count, dtype: int64

**Findings**:
- The per-attribute floor alone only needed **23** distinct clusters to give all 11 attributes
  their minimum of 8 — far below the 88 it would take with zero overlap. Confirms the geo/name
  attributes' conflicts co-occur heavily in the same clusters.
- The 102-cluster global top-up (to reach 125) pushed most attributes well past their floor "for
  free": `lat` 124/125, `city` 124/125, `lon` 123/125, `name` 120/125, `countryName` 101/125,
  `timezone` 89/125. The attributes that stayed near the floor don't co-occur with these as much:
  `localCode` 8/125 (exactly the minimum — its disagreements look largely independent of the
  others), `iata` 18/125, `icao` 26/125, `elevation` 54/125, `countryCode` 58/125.
- `source_combination` within the final 125 is skewed hard toward `wd+oa+kg` (103, 82%) vs.
  `wd+oa` (18) and `oa+kg` (4) — far more than their share of all clusters (22% / 75% / 3%
  respectively). Expected: a 3-way cluster has more attribute-pairs that can disagree than a
  2-way one, so it scores higher on `n_disagree_attrs` and wins more top-up slots. This is fine —
  the goal here is conflict coverage, not a population-representative sample — but worth knowing:
  this gold standard will mostly exercise 3-source fusion, with only 18 + 4 = 22 two-source cases.
- `localCode` and `timezone` each only have 2 of the 3 `source_combination`s available at all
  (`localCode`: no `oa+kg`, since Kaggle has no `localCode` column; `timezone`: only `wd+oa+kg`,
  since it needs both `wd` and `kg` present simultaneously) — not a bug, just their real source
  coverage.


## Part 3b — Manual Inspection (done by you, outside this notebook's logic)

For each `cluster_id` in `candidates_final`, use `inspect_cluster` below to see every attribute
from every source side by side, confirm the three records really are the same airport (this also
catches any wrong correspondence from the identity-resolution step), and check/correct the ICAO.
Collect the final `(id, icao)` pairs — `id` is that cluster's `oa_id` — for Part 4.

In [15]:
def inspect_cluster(cluster_id: int) -> pd.DataFrame:
    """Side-by-side raw values (all columns) for one cluster's records, one column per source."""
    row = clusters.set_index("cluster_id").loc[cluster_id]
    cols = {}
    for src in ["wd", "oa", "kg"]:
        rec_id = row[f"{src}_id"]
        if pd.isna(rec_id):
            continue
        cols[src] = SOURCE_DFS[src].set_index("id").loc[rec_id]
    return pd.DataFrame(cols)


# example: first candidate cluster
inspect_cluster(candidates_final.iloc[0]["cluster_id"])

,wd,oa,kg
city,milne bay province,misima island,misima island
cityIata,NaN,NaN,MIS
continent,NaN,OC,NaN
countryCode,PG,PG,PG
countryCodeA3,NaN,NaN,PNG
countryName,papua new guinea,papua new guinea,papua new guinea
elevation,27.0,7.9248,NaN
iata,MIS,MIS,MIS
icao,AYMS,AYMS,AYMS
isoRegion,NaN,PG-MBA,NaN


In [16]:
'''
Collect the REAL ICAO ( after manual check ) and write as a list of (id, icao)

Ex : 

demo_pairs = [
    ("oa_12269", "AYMS"),   # Misima Island Airport, PNG
    ("oa_19944", "ZBNY"),   # Beijing Nanyuan Airport, CN
    ("oa_38630", "KBFZ"),   # Albertville Regional Airport, US
]
'''


'\nCollect the REAL ICAO ( after manual check ) and write as a list of (id, icao)\n\nEx : \n\ndemo_pairs = [\n    ("oa_12269", "AYMS"),   # Misima Island Airport, PNG\n    ("oa_19944", "ZBNY"),   # Beijing Nanyuan Airport, CN\n    ("oa_38630", "KBFZ"),   # Albertville Regional Airport, US\n]\n'

In [31]:
### For now I just demo first, but we still need to verify the ICAO manually

from collections import Counter


def majority_icao(row):
    """Most-voted non-null ICAO among icao_wd/icao_oa/icao_kg; NaN if all three are missing."""
    votes = [row.get("icao_wd"), row.get("icao_oa"), row.get("icao_kg")]
    votes = [v for v in votes if pd.notna(v)]
    if not votes:
        return float("nan")
    return Counter(votes).most_common(1)[0][0]


demo_rows = candidates_final.head(3)
majority_pairs = [
    (row["oa_id"], majority_icao(row)) for _, row in candidates_final.iterrows()
]
majority_pairs[:3]


[('oa_12269', 'AYMS'), ('oa_19944', 'ZBNY'), ('oa_20622', nan)]

## Part 4 — Look Up External Ground Truth, Write `gold.xml`

**Goal**: given the `(id, icao)` pairs you confirm by hand in Part 3b, look up each airport's
true values for the 10 target fields from an external, independent source (OpenStreetMap), and
write them out as `gold.xml`.

**Method** — three different lookups chained together, as discussed:
1. **Overpass API** — one batched query for the whole input list (`nwr["aeroway"~"aerodrome|heliport"]["icao"="..."]`
   unioned together), giving `name`, `iata`, `icao`, `lat`, `lon`, `elevation` (OSM `ele` tag) in a
   single request. `localCode` is also attempted here (`faa`/`ref` tags) but OSM has no reliable
   equivalent, so expect it to come back empty often.
2. **Nominatim reverse geocoding** — one request per airport (no batch endpoint), using the
   lat/lon from step 1, rate-limited to ~1 req/sec per Nominatim's usage policy. Gives
   `countryCode`, `countryName`, `city`. ⚠️ Spot-check `city`: reverse geocoding returns the
   administrative locality the coordinates fall in, which often isn't the city aviation databases
   conventionally list (e.g. Dulles Airport's coordinates are in Virginia, not "Washington").
3. **`timezonefinder`** — local/offline lookup from lat/lon, no network call, no rate limit.

**Output**: `<Airports><Airport id="...">...</Airport>...</Airports>`, element order and names
matching `integrated_schema.xsd` (`name, iata, icao, localCode, lat, lon, elevation, countryCode,
countryName, city, timezone`); a missing value is kept as an empty element rather than omitted.

In [32]:
import datetime
import time
import xml.etree.ElementTree as ET
from xml.dom import minidom

import requests
from timezonefinder import TimezoneFinder

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
NOMINATIM_URL = "https://nominatim.openstreetmap.org/reverse"
USER_AGENT = "wdi-airport-integration-project (student coursework; contact via GitHub repo)"
NOMINATIM_SLEEP_SECONDS = 1.1  # Nominatim policy: max ~1 req/sec

XML_FIELD_ORDER = [
    "name", "iata", "icao", "localCode", "lat", "lon",
    "elevation", "countryCode", "countryName", "city", "timezone",
]

_tf = TimezoneFinder()

### Step 1 — Overpass: name, iata, icao, localCode, lat, lon, elevation

In [33]:
def fetch_overpass_batch(icao_codes: list[str], timeout: int = 60) -> dict[str, dict]:
    """One Overpass request for every ICAO code in the list."""
    codes = sorted({c.upper() for c in icao_codes if c})
    if not codes:
        return {}

    clauses = "\n".join(
        f'  nwr["aeroway"~"^(aerodrome|heliport)$"]["icao"="{code}"];' for code in codes
    )
    query = f"[out:json][timeout:{timeout}];\n(\n{clauses}\n);\nout center tags;"

    resp = None
    for attempt in range(3):
        resp = requests.post(
            OVERPASS_URL,
            data={"data": query},
            headers={"User-Agent": USER_AGENT},
            timeout=timeout + 10,
        )
        if resp.ok:
            break
        wait = 2 ** (attempt + 1)  # Overpass's public instance occasionally 502s under load
        print(f"Overpass returned {resp.status_code}, retrying in {wait}s (attempt {attempt + 1}/3)")
        time.sleep(wait)
    resp.raise_for_status()
    elements = resp.json().get("elements", [])

    by_icao: dict[str, list[dict]] = {}
    for el in elements:
        tags = el.get("tags", {})
        icao = tags.get("icao")
        if not icao:
            continue
        by_icao.setdefault(icao, []).append(el)

    dupes = {icao: els for icao, els in by_icao.items() if len(els) > 1}
    if dupes:
        print(f"Overpass: {len(dupes)} ICAO code(s) matched more than one OSM object "
              "(data-quality issue in OSM, not in our query) \u2014 taking the first, "
              "but worth a manual look before trusting it:")
        for icao, els in dupes.items():
            names = [e.get("tags", {}).get("name") for e in els]
            print(f"  {icao}: {names}")

    results: dict[str, dict] = {}
    for icao, els in by_icao.items():
        el = els[0]
        tags = el.get("tags", {})
        center = el.get("center", {})
        results[icao] = {
            "name": tags.get("name:en") or tags.get("name"),
            "iata": tags.get("iata"),
            "icao": icao,
            "localCode": tags.get("faa") or tags.get("ref"),
            "lat": el.get("lat", center.get("lat")),
            "lon": el.get("lon", center.get("lon")),
            "elevation": tags.get("ele"),
        }

    missing = set(codes) - results.keys()
    if missing:
        print(f"Overpass: no match for {len(missing)} of {len(codes)} codes: {sorted(missing)}")
    return results


### Step 2 — Nominatim reverse geocoding: countryCode, countryName, city

In [34]:
def reverse_geocode(lat, lon) -> dict:
    """One Nominatim reverse-geocoding call. Rate-limited: sleeps before every call."""
    if lat is None or lon is None:
        return {}

    time.sleep(NOMINATIM_SLEEP_SECONDS)
    resp = requests.get(
        NOMINATIM_URL,
        params={
            "format": "jsonv2",
            "lat": lat,
            "lon": lon,
            "zoom": 10,
            "addressdetails": 1,
            "accept-language": "en",  # keep countryName/city consistent with the rest of the project
        },
        headers={"User-Agent": USER_AGENT},
        timeout=30,
    )
    resp.raise_for_status()
    address = resp.json().get("address", {})

    city = (
        address.get("city") or address.get("town") or address.get("village")
        or address.get("municipality") or address.get("county")
    )
    country_code = address.get("country_code")
    return {
        "countryCode": country_code.upper() if country_code else None,
        "countryName": address.get("country"),
        "city": city,
    }


### Step 3 — `timezonefinder`: timezone (local, no API call)

In [35]:
def lookup_timezone(lat, lon):
    if lat is None or lon is None:
        return None
    return _tf.timezone_at(lat=float(lat), lng=float(lon))

### Orchestration + XML writer

In [36]:
def build_gold_entries(id_icao_pairs: list[tuple[str, str]]) -> list[dict]:
    """id_icao_pairs: [(id, icao), ...], id = the oa_id picked in Part 3b."""
    icao_codes = [icao for _, icao in id_icao_pairs]
    osm = fetch_overpass_batch(icao_codes)
    fetched_at = datetime.datetime.now(datetime.timezone.utc).isoformat()

    entries = []
    for record_id, icao in id_icao_pairs:
        icao = icao.upper()
        base = osm.get(icao, {"icao": icao})
        lat, lon = base.get("lat"), base.get("lon")

        geo = reverse_geocode(lat, lon)
        tz = lookup_timezone(lat, lon)

        entries.append({
            "id": record_id,
            "name": base.get("name"),
            "iata": base.get("iata"),
            "icao": icao,
            "localCode": base.get("localCode"),
            "lat": lat,
            "lon": lon,
            "elevation": base.get("elevation"),
            "countryCode": geo.get("countryCode"),
            "countryName": geo.get("countryName"),
            "city": geo.get("city"),
            "timezone": tz,
            "_provenance": {
                "fetched_at": fetched_at,
                "overpass_matched": icao in osm,
                "nominatim_queried": lat is not None and lon is not None,
            },
        })
    return entries


def write_gold_xml(entries: list[dict], path) -> None:
    root = ET.Element("Airports")
    for entry in entries:
        airport = ET.SubElement(root, "Airport", id=str(entry["id"]))
        for field in XML_FIELD_ORDER:
            value = entry.get(field)
            el = ET.SubElement(airport, field)
            el.text = "" if value is None else str(value)

    pretty = minidom.parseString(ET.tostring(root, encoding="utf-8")).toprettyxml(
        indent="  ", encoding="utf-8"
    )
    Path(path).write_bytes(pretty)

### Full run — all 125 clusters' majority-vote ICAO (no manual verification yet)

You decided to go with the majority-vote ICAO across all 125 candidate clusters rather than
manually confirming each one first. Rows where all three sources are missing ICAO are dropped
(can't look anything up without an ICAO). This hits the real Overpass + Nominatim APIs for every
remaining row, so it takes a few minutes (~1.1s/row just for Nominatim's rate limit).


In [45]:
import json


demo_1 = [i for i in majority_pairs if not pd.isna(i[1])] 
# we dont need this demo_1 after we finish part3b (manual inspection), since every icao has value

demo_entries = build_gold_entries(demo_1)

for e in demo_entries:
    print(e)

Overpass returned 502, retrying in 2s (attempt 1/3)
Overpass: 5 ICAO code(s) matched more than one OSM object (data-quality issue in OSM, not in our query) — taking the first, but worth a manual look before trusting it:
  KSAW: ['Marquette Sawyer Regional Airport', 'Marquette Sawyer Regional Airport']
  HHOH: ['Om Hajer', 'Umm Hajar أم هاجر']
  MNPC: ['Aeropuerto de Bilwi', 'Aeropuerto de Bilwi']
  ZBNY: ['北京南苑机场', '北京南郊机场']
  NTAA: ["GAM FAA'A", "Aéroport international de Tahiti-Faa'a"]
Overpass: no match for 11 of 117 codes: ['HCMW', 'K8A0', 'KCTJ', 'KL08', 'LKBL', 'Q51', 'SFPH', 'SMDO', 'UGSS', 'UKFB', 'XLLL']
{'id': 'oa_12269', 'name': 'Misima Island Airport', 'iata': 'MIS', 'icao': 'AYMS', 'localCode': None, 'lat': -10.6891318, 'lon': 152.8377488, 'elevation': '27', 'countryCode': 'PG', 'countryName': 'Papua New Guinea', 'city': 'Samarai-Murua District', 'timezone': 'Pacific/Port_Moresby', '_provenance': {'fetched_at': '2026-11-14T15:32:08.662762+00:00', 'overpass_matched': True, 

In [46]:
write_gold_xml(demo_entries, GOLD_DIR / "gold_standard.xml")
print(f"Saved {len(demo_entries)} airports to {GOLD_DIR / 'gold_standard.xml'}")


Saved 117 airports to ../../data/data_fusion/golden_standard/gold_standard.xml
